In [8]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

print(ground_truth)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {'id': '489dd1c9d9', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?', 'answer': 'The zoom link is only pu

In [10]:
doc_idx = {}

for doc in documents:
    doc_idx[doc['id']] = doc

In [11]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

In [12]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ Database for entries matching the given query
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [16]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

load_dotenv()

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [17]:
rec = ground_truth[0]
result = runner.loop(prompt=rec["question"])

In [19]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='I found this course late — can I still sign up and follow along?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"late enrollment sign up follow along course found late"}', call_id='call_GiAqp4mfTlRntZ4MigyvcFYZ', name='search', type='function_call', id='fc_095417680e435274006a4ccd52fd5c819abdf67f43877820be', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_GiAqp4mfTlRntZ4MigyvcFYZ',
  'output': '[\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoo

In [20]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue
            
        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments
            })
    
    return tool_calls

In [21]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"late enrollment sign up follow along course found late"}'}]

In [23]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [24]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id
}

agent_result

{'question': 'I found this course late — can I still sign up and follow along?',
 'answer_agent': 'Yes — you can start whenever you want and follow along at your own pace.\n\nThe course videos, docs, and GitHub materials are available, and you can work through the lessons and homework as you go. Just note that homework can only be submitted while the submission form is open; there are no late submissions once it closes.\n\nOne important caveat: if you want a certificate, you need to finish with the live cohort rather than purely self-paced.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"late enrollment sign up follow along course found late"}'}],
 'cost': Decimal('0.001311'),
 'document': '74eb249bbf'}

In [25]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id
    }

    return answer_record

In [26]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [40]:
df_agent = pd.DataFrame(agent_answers)

In [30]:
df_agent["cost"].sum()

Decimal('0.06344400')

In [31]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [32]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

In [33]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct"
    )

    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer"
    )

    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )

    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [34]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [41]:
import ast, json
from evaluation_utils import calc_total_price, llm_structured_retry

openai_client = OpenAI()

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    print(tool_calls)

    if isinstance(tool_calls, str):
        try:
            tool_calls = json.loads(tool_calls)
        except json.JSONDecodeError:
            tool_calls = ast.literal_eval(tool_calls)
    
    prompt = agent_judge_prompt.format(
        question = rec["question"],
        answer_orig = rec["answer_orig"],
        answer_agent = rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model
    )

    return result, usage

In [42]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])
agent_eval

[{'name': 'search', 'arguments': '{"query":"found course late can I still sign up and follow along late enrollment join after start"}'}]


AgentEvaluation(answer_reasoning='The agent’s answer matches the ground truth. It says the user can still join late and follow along, and it includes the key condition that to receive a certificate they must submit the project while submissions are still being accepted.', answer_score='good', trajectory_reasoning="The single search query is relevant and includes important keywords from the question like 'found course late' and 'can I still sign up and follow along'. One search is sufficient here, and there were no unnecessary duplicate calls.", trajectory_score='good')

In [43]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning
    }

    return result, usage

In [44]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

[{'name': 'search', 'arguments': '{"query":"found course late can I still sign up and follow along late enrollment join after start"}'}]
[{'name': 'search', 'arguments': '{"query":"too late to join course heard about it now enrollment late join"}'}]
[{'name': 'search', 'arguments': '{"query":"Can I still take part in the course even after it already started? late enrollment started course"}'}]
[{'name': 'search', 'arguments': '{"query":"late join certificate attendance certificate joining late certificate still able to get certificate"}'}]
[{'name': 'search', 'arguments': '{"query":"certificate eligible started course late late start eligibility certificate requirements"}'}]
[{'name': 'search', 'arguments': '{"query":"LLM Zoomcamp confirmation email registered signed up no confirmation email do I still count as registered"}'}]
[{'name': 'search', 'arguments': '{"query":"acceptance confirmation message start course wait before can start course"}'}]
[{'name': 'search', 'arguments': '{"qu

In [45]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [46]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [47]:
calc_total_price(usages)

0.05371649999999999

In [49]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    47
bad      3
Name: count, dtype: int64

In [50]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    49
bad      1
Name: count, dtype: int64

In [51]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)

In [ ]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]